# Load data 

In [0]:
import pandas as pd
import numpy as np

fe = pd.read_parquet("/Volumes/workspace/default/retail_data/customer_features_engineered.parquet")

print(fe.shape)
fe.head()

(5878, 15)


,Customer ID,total_spent,avg_unit_price,num_invoices,num_line_items,num_unique_products,num_countries,recency_days,active_days,purchase_frequency,avg_basket_value,log_total_spent,log_num_invoices,log_num_unique_products,log_avg_basket_value
0,17764,1623.64,3.16,5,159,126,1,50,687,0.0073,324.73,7.393042,1.791759,4.844187,5.786069
1,15031,3156.58,2.83,14,405,237,1,4,728,0.0192,225.47,8.057561,2.708050,5.472271,5.422612
2,15785,5969.93,2.68,16,225,102,1,32,693,0.0231,373.12,8.694658,2.833213,4.634729,5.924577
3,18041,8752.38,2.05,38,960,464,1,11,684,0.0556,230.33,9.077195,3.663562,6.142037,5.443845
4,14344,3333.70,3.58,20,243,140,1,127,500,0.0400,166.69,8.112138,3.044522,4.948760,5.122117


# Cell 2 - Rule-based Segmentation


In [0]:

# Tính percentile thresholds dựa trên 3 trục RFM
r_33 = fe["recency_days"].quantile(0.33)
r_66 = fe["recency_days"].quantile(0.66)

f_33 = fe["num_invoices"].quantile(0.33)
f_66 = fe["num_invoices"].quantile(0.66)

m_33 = fe["total_spent"].quantile(0.33)
m_66 = fe["total_spent"].quantile(0.66)

print("=== THRESHOLDS ===")
print(f"Recency   : 33%={r_33:.0f} days | 66%={r_66:.0f} days")
print(f"Frequency : 33%={f_33:.0f} invoices | 66%={f_66:.0f} invoices")
print(f"Monetary  : 33%=£{m_33:.0f} | 66%=£{m_66:.0f}")

# RFM Score - mỗi trục cho 1 2 3
# Recency: càng nhỏ càng tốt → reverse
fe["R"] = pd.cut(fe["recency_days"],
                 bins=[-1, r_33, r_66, float("inf")],
                 labels=[3, 2, 1]).astype(int)

fe["F"] = pd.cut(fe["num_invoices"],
                 bins=[0, f_33, f_66, float("inf")],
                 labels=[1, 2, 3]).astype(int)

fe["M"] = pd.cut(fe["total_spent"],
                 bins=[0, m_33, m_66, float("inf")],
                 labels=[1, 2, 3]).astype(int)

fe["RFM_score"] = fe["R"] + fe["F"] + fe["M"]

# Gán segment
def assign_segment(score):
    if score >= 8:
        return "Champions"
    elif score >= 6:
        return "Loyal"
    elif score >= 4:
        return "Potential"
    else:
        return "At Risk"

fe["segment"] = fe["RFM_score"].apply(assign_segment)

print("\n=== SEGMENT DISTRIBUTION ===")
print(fe["segment"].value_counts())
print(f"\n% distribution:")
print(fe["segment"].value_counts(normalize=True).round(3) * 100)

=== THRESHOLDS ===
Recency   : 33%=40 days | 66%=260 days
Frequency : 33%=2 invoices | 66%=5 invoices
Monetary  : 33%=£466 | 66%=£1580

=== SEGMENT DISTRIBUTION ===
segment
Potential    1682
Champions    1681
Loyal        1424
At Risk      1091
Name: count, dtype: int64

% distribution:
segment
Potential    28.6
Champions    28.6
Loyal        24.2
At Risk      18.6
Name: proportion, dtype: float64


## Customer Segmentation - RFM Rule-based

### Thresholds:
| Trục | Low (score=1) | Mid (score=2) | High (score=3) |
|---|---|---|---|
| Recency | > 260 ngày | 40–260 ngày | < 40 ngày |
| Frequency | ≤ 2 invoices | 3–5 invoices | > 5 invoices |
| Monetary | < £466 | £466–£1,580 | > £1,580 |

### Segment Distribution:
| Segment | Count | % | Mô tả |
|---|---|---|---|
| **Champions** | 1,681 | 28.6% | Mua gần đây, thường xuyên, chi nhiều |
| **Loyal** | 1,424 | 24.2% | Ổn định, gắn bó lâu dài |
| **Potential** | 1,682 | 28.6% | Tiềm năng, cần nurture thêm |
| **At Risk** | 1,091 | 18.6% | Lâu không mua, có nguy cơ churn |

### Nhận xét:
- Phân bổ khá đều → model không bị imbalanced nặng
- **At Risk 18.6%** → đây là nhóm business cần action ngay:
  remarketing, discount, win-back campaign
- **Champions + Loyal = 52.8%** → hơn nửa customer base đang healthy

# CELL - 3 Save data

In [0]:
fe.to_parquet("/Volumes/workspace/default/retail_data/customer_labeled.parquet", index=False)
print("Saved!")

Saved!
